# GNN Fraud Detection (IBM AML Data)

Graph Neural Network for transaction fraud. Nodes = transactions, edges link shared accounts; we predict **Is Laundering** per transaction.

In [1]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from torch.nn import Linear
import torch.nn.functional as F
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, average_precision_score

In [2]:
# Data path: use kagglehub cache or set DATA_PATH to your CSV
PROJECT_ROOT = Path.cwd()
CACHE = PROJECT_ROOT / "kagglehub_cache" / "datasets" / "ealtman2019" / "ibm-transactions-for-anti-money-laundering-aml" / "versions" / "8"
FILE_NAME = "HI-Small_Trans.csv"
DATA_PATH = CACHE / FILE_NAME
if not DATA_PATH.exists():
    DATA_PATH = PROJECT_ROOT / FILE_NAME  # fallback: same folder as notebook
df = pd.read_csv(DATA_PATH)
print(df.shape)
print(df.head())

(5078345, 11)
          Timestamp  From Bank    Account  To Bank  Account.1  \
0  2022/09/01 00:20         10  8000EBD30       10  8000EBD30   
1  2022/09/01 00:20       3208  8000F4580        1  8000F5340   
2  2022/09/01 00:00       3209  8000F4670     3209  8000F4670   
3  2022/09/01 00:02         12  8000F5030       12  8000F5030   
4  2022/09/01 00:06         10  8000F5200       10  8000F5200   

   Amount Received Receiving Currency  Amount Paid Payment Currency  \
0          3697.34          US Dollar      3697.34        US Dollar   
1             0.01          US Dollar         0.01        US Dollar   
2         14675.57          US Dollar     14675.57        US Dollar   
3          2806.97          US Dollar      2806.97        US Dollar   
4         36682.97          US Dollar     36682.97        US Dollar   

  Payment Format  Is Laundering  
0   Reinvestment              0  
1         Cheque              0  
2   Reinvestment              0  
3   Reinvestment              0 

In [3]:
# Feature prep: timestamp + encode categoricals
ts = pd.to_datetime(df["Timestamp"], errors="coerce")
df = df.copy()
df["Hour"] = ts.dt.hour.fillna(-1).astype("int16")
df["DayOfWeek"] = ts.dt.dayofweek.fillna(-1).astype("int16")
df["Day"] = ts.dt.day.fillna(-1).astype("int16")
df["Month"] = ts.dt.month.fillna(-1).astype("int16")
for col in ["Account", "Account.1", "Receiving Currency", "Payment Currency", "Payment Format"]:
    if col in df.columns:
        df[col] = pd.factorize(df[col], sort=True)[0].astype("int32")
y = df["Is Laundering"].astype(int).values
feature_cols = ["From Bank", "Account", "To Bank", "Account.1", "Amount Received", "Receiving Currency",
                "Amount Paid", "Payment Currency", "Payment Format", "Hour", "DayOfWeek", "Day", "Month"]
feature_cols = [c for c in feature_cols if c in df.columns]
X = df[feature_cols].fillna(0).astype(np.float32)
print("Features:", feature_cols)
print("Label balance:", pd.Series(y).value_counts())

Features: ['From Bank', 'Account', 'To Bank', 'Account.1', 'Amount Received', 'Receiving Currency', 'Amount Paid', 'Payment Currency', 'Payment Format', 'Hour', 'DayOfWeek', 'Day', 'Month']
Label balance: 0    5073168
1       5177
Name: count, dtype: int64


In [4]:
# Optional subsample for quick runs
MAX_ROWS = 150_000
if len(df) > MAX_ROWS:
    idx = df.groupby("Is Laundering", group_keys=False).apply(
        lambda g: g.sample(frac=MAX_ROWS/len(df), random_state=42)
    ).index
    df = df.loc[idx].reset_index(drop=True)
    X = X.loc[idx].reset_index(drop=True)
    y = df["Is Laundering"].astype(int).values
print("Using", len(df), "rows")

Using 150000 rows


In [5]:
# Build graph: nodes = transactions (rows), edges = same account connects two transactions
# Edge index: for each (sender, receiver) account pair, connect transactions that share an account
try:
    from torch_geometric.data import Data
    from torch_geometric.nn import GCNConv, global_mean_pool
except ImportError:
    raise ImportError("Install: pip install torch torch-geometric")

n_nodes = len(df)
account_to_txs = {}  # account_id -> list of transaction (row) indices
for pos in range(len(df)):
    row = df.iloc[pos]
    a1, a2 = row["Account"], row["Account.1"]
    for a in (a1, a2):
        account_to_txs.setdefault(a, []).append(pos)

edge_src, edge_dst = [], []
for txs in account_to_txs.values():
    for i in range(len(txs)):
        for j in range(i + 1, min(i + 6, len(txs))):  # limit neighbors per account
            edge_src.append(txs[i])
            edge_dst.append(txs[j])
            edge_src.append(txs[j])
            edge_dst.append(txs[i])

edge_index = torch.tensor([edge_src, edge_dst], dtype=torch.long) if edge_src else torch.zeros((2, 0), dtype=torch.long)
x = torch.from_numpy(X.values)
y_t = torch.from_numpy(y).long()
data = Data(x=x, edge_index=edge_index, y=y_t)
print("Nodes:", data.num_nodes, "Edges:", data.edge_index.shape[1])

c:\Users\phine\OneDrive\Laptop\CODESTUFF\HACKATHON\genai\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Nodes: 150000 Edges: 519968


C:\Users\phine\AppData\Local\Temp\ipykernel_29404\4190432208.py:28: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\utils\tensor_numpy.cpp:219.)
  y_t = torch.from_numpy(y).long()


In [6]:
class GCN(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super().__init__()
        self.conv1 = GCNConv(in_channels, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, out_channels)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index).relu()
        x = F.dropout(x, p=0.3, training=self.training)
        x = self.conv2(x, edge_index)
        return x

In [7]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
train_idx, test_idx = train_test_split(np.arange(n_nodes), test_size=0.2, random_state=42, stratify=y)
train_mask = torch.zeros(n_nodes, dtype=torch.bool)
train_mask[train_idx] = True
test_mask = torch.zeros(n_nodes, dtype=torch.bool)
test_mask[test_idx] = True
data.train_mask = train_mask
data.test_mask = test_mask
data = data.to(device)

in_dim = X.shape[1]
model = GCN(in_dim, 64, 2).to(device)
opt = torch.optim.Adam(model.parameters(), lr=0.01)
weights = torch.tensor([1.0, (data.y[train_mask] == 0).sum().float() / max((data.y[train_mask] == 1).sum().item(), 1)], device=device)
criterion = torch.nn.CrossEntropyLoss(weight=weights)

In [8]:
def train():
    model.train()
    opt.zero_grad()
    out = model(data.x, data.edge_index)
    loss = criterion(out[data.train_mask], data.y[data.train_mask])
    loss.backward()
    opt.step()
    return loss.item()

for epoch in range(1, 201):
    loss = train()
    if epoch % 50 == 0:
        print(f"Epoch {epoch}, loss={loss:.4f}")

Epoch 50, loss=19158.7324
Epoch 100, loss=692.0547
Epoch 150, loss=250.6949
Epoch 200, loss=183.2233


In [9]:
model.eval()
with torch.no_grad():
    out = model(data.x, data.edge_index)
    pred = out.argmax(dim=1)
    prob = F.softmax(out, dim=1)[:, 1]

y_test = data.y[data.test_mask].cpu().numpy()
y_pred = pred[data.test_mask].cpu().numpy()
y_prob = prob[data.test_mask].cpu().numpy()

print("Classification Report:")
print(classification_report(y_test, y_pred, digits=4, zero_division=0))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print(f"\nROC-AUC: {roc_auc_score(y_test, y_prob):.4f}")
print(f"PR-AUC:  {average_precision_score(y_test, y_prob):.4f}")

Classification Report:
              precision    recall  f1-score   support

           0     0.9994    0.7836    0.8785     29969
           1     0.0025    0.5161    0.0049        31

    accuracy                         0.7834     30000
   macro avg     0.5009    0.6499    0.4417     30000
weighted avg     0.9983    0.7834    0.8776     30000

Confusion Matrix:
[[23485  6484]
 [   15    16]]

ROC-AUC: 0.6519
PR-AUC:  0.0020
